In [1]:
import sys
!~/.local/bin/uv pip install --python {sys.executable} requests torch numpy matplotlib --quiet

In [2]:
import struct
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import requests

In [ ]:
BUTTON_BITS = [
    0x8000, 0x4000, 0x2000, 0x1000,
    0x0800, 0x0400, 0x0200, 0x0100,
    0x0020, 0x0010, 0x0008, 0x0004, 0x0002, 0x0001,
]

# 6 output categories
CAT_NAMES = ["Control Stick", "Letter Buttons (A/B/Z/L/R)", "Start", "D-Pad", "C-Buttons", "No Input"]
INPUT_DIM  = 6

# Which columns of the 16-col base tensor [stick_x, stick_y, A..CRight] map to each category
#   col 0  = stick_x   col 1  = stick_y
#   col 2  = A         col 3  = B
#   col 4  = Z         col 5  = Start
#   col 6  = DUp       col 7  = DDown    col 8  = DLeft   col 9  = DRight
#   col 10 = L         col 11 = R
#   col 12 = CUp       col 13 = CDown    col 14 = CLeft   col 15 = CRight
CAT_COLS = [
    [0, 1],              # Control Stick
    [2, 3, 4, 10, 11],   # Letter Buttons (A/B/Z/L/R)
    [5],                 # Start
    [6, 7, 8, 9],        # D-Pad
    [12, 13, 14, 15],    # C-Buttons
]


def parse_m64(raw: bytes) -> torch.Tensor:
    """Parse .m64 bytes into a (T, 6) binary float32 tensor — one column per category.
    Categories: [Control Stick, Letter Buttons, Start, D-Pad, C-Buttons, No Input]
    """
    version = struct.unpack_from("<I", raw, 0x004)[0]
    data_start = 0x200 if version in (1, 2) else 0x400
    num_samples = struct.unpack_from("<I", raw, 0x018)[0]

    buf = np.frombuffer(
        raw[data_start : data_start + num_samples * 4], dtype=np.uint8
    ).reshape(num_samples, 4)

    y_axis  = buf[:, 0].view(np.int8)
    x_axis  = buf[:, 1].view(np.int8)
    btn_raw = buf[:, 2].astype(np.uint16) * 256 + buf[:, 3].astype(np.uint16)
    buttons = np.stack(
        [(btn_raw & bit).astype(bool) for bit in BUTTON_BITS], axis=1
    ).astype(np.float32)

    stick_x = torch.from_numpy((x_axis != 0).astype(np.float32)).unsqueeze(1)
    stick_y = torch.from_numpy((y_axis != 0).astype(np.float32)).unsqueeze(1)
    btns    = torch.from_numpy(buttons)
    base    = torch.cat([stick_x, stick_y, btns], dim=1)  # (T, 16)

    cat_active = torch.stack(
        [base[:, cols].any(dim=1) for cols in CAT_COLS], dim=1
    ).float()  # (T, 5)

    no_input = (~cat_active.any(dim=1)).unsqueeze(1).float()
    return torch.cat([cat_active, no_input], dim=1)  # (T, 6)


def load_m64(source: str, cache_dir: str = ".") -> torch.Tensor:
    """Load an .m64 from a local path or URL, caching downloads locally."""
    if source.startswith("http://") or source.startswith("https://"):
        cache_path = Path(cache_dir) / Path(source).name
        if not cache_path.exists():
            print(f"Downloading {source} ...")
            cache_path.write_bytes(requests.get(source).content)
        raw = cache_path.read_bytes()
    else:
        raw = Path(source).read_bytes()
    return parse_m64(raw)

In [4]:
def expand_to_sequential(frames: torch.Tensor) -> torch.Tensor:
    """Convert (T, 7) multi-hot frames to (N, 7) sequential one-hot events.

    Each active category in a frame becomes its own entry in column order.
    A frame with Control Stick + A/B active → two consecutive one-hot vectors.
    A no-input frame → one one-hot vector at col 6.
    """
    events = []
    eye = torch.eye(INPUT_DIM)
    for frame in frames:
        for col in frame.nonzero(as_tuple=True)[0]:
            events.append(eye[col])
    return torch.stack(events)  # (N, 7)


class M64Dataset(Dataset):
    """Sequential one-hot event dataset over one or more .m64 files.

    Each item is a (7,) one-hot float32 vector representing a single input category event.
    Simultaneous categories in the same frame are expanded into consecutive entries
    in column order: [Control Stick, A/B, Z/L/R, Start, D-Pad, C-Buttons, No Input].

    Args:
        sources:   list of local paths or URLs to .m64 files.
        cache_dir: directory to cache downloaded files.
    """

    def __init__(self, sources: list[str], cache_dir: str = "."):
        self.file_names: list[str] = []
        events = []

        for file_idx, src in enumerate(sources):
            print(f"[{file_idx}] Loading: {src}")
            raw_frames = load_m64(src, cache_dir=cache_dir)
            events.append(expand_to_sequential(raw_frames))
            self.file_names.append(Path(src).name)

        self.data = torch.cat(events, dim=0)  # (total_events, 7)
        print(f"\nTotal events: {len(self.data):,}")

    def __len__(self) -> int:
        return len(self.data)

    def __getitem__(self, idx: int) -> torch.Tensor:
        return self.data[idx]  # (7,)

In [5]:
SOURCES = [
    "./120_stars.m64",
    "./70_stars.m64",
    "./big_bob_omb_on_the_summit.m64"
]

BATCH_SIZE = 128

dataset = M64Dataset(SOURCES)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

print(f"Batches per epoch: {len(loader):,}")

[0] Loading: ./120_stars.m64
[1] Loading: ./70_stars.m64
[2] Loading: ./big_bob_omb_on_the_summit.m64

Total events: 614,175
Batches per epoch: 4,799


In [ ]:
# Inspect one batch
batch = next(iter(loader))
print("batch shape:", batch.shape)   # (B, 6)
print("first event:", batch[0])
print()
print("Column layout:")
for i, name in enumerate(CAT_NAMES):
    print(f"  col {i} = {name}")

In [ ]:
import matplotlib.pyplot as plt

def plot_activity(tensor, title):
    counts = tensor.sum(dim=0)
    pct    = counts / counts.sum() * 100

    # Sort ascending so highest bar is on the right
    order  = pct.argsort()
    names  = [CAT_NAMES[i] for i in order]
    values = pct[order].numpy()

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(names, values, color="steelblue", alpha=0.85)
    ax.set_xticklabels(names, rotation=45, ha="right")
    ax.set_ylabel("% of total input events")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

offset = 0
for name, src in zip(dataset.file_names, SOURCES):
    n = expand_to_sequential(load_m64(src)).shape[0]
    plot_activity(dataset.data[offset : offset + n], name)
    offset += n

plot_activity(dataset.data, "Combined (all files)")